# Voilà app: smooth client-side fit explorer

This version keeps the **simulation in Python**, but moves the **fit slider updates entirely into the browser**.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import json, uuid, html
import numpy as np
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks
from numba import njit
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output


In [ ]:

@njit
def dphi_dt(t, phi, p, q, p_s, q_s):
    L = len(phi)
    dphi = np.zeros(L)
    vp = p * (1 - phi[0]) + p_s * phi[0]
    vm = q * (1 - phi[L - 1]) + q_s * phi[L - 1]
    dphi[1:L-1] = (
        phi[:-2] - 2 * phi[1:-1] + phi[2:] + vp * (phi[2:L] - phi[1:L-1]) - vm * (phi[1:L-1] - phi[0:L-2])
    )
    dphi[0] = ((1 - phi[0]) * (q_s * phi[L - 1] + (1 + p) * phi[1]) - (1 + p_s) * phi[0] * (1 - phi[1]) - q * phi[0] * (1 - phi[L - 1]))
    dphi[L - 1] = ((1 - phi[L - 1]) * (p_s * phi[0] + (1 + q) * phi[L - 2]) - (1 + q_s) * phi[L - 1] * (1 - phi[L - 2]) - p * phi[L - 1] * (1 - phi[0]))
    return dphi

def get_rho_crit(p, q, p_s, q_s):
    denom = (p * q_s) - (q * p_s)
    if abs(denom) < 1e-12:
        return np.nan, np.nan
    return (q_s * (p - q)) / denom, (p_s * (p - q)) / denom

def model_curve(z, t, v0, A, s):
    arg = v0 + (2 * z / 3)
    out = np.full_like(z, np.nan, dtype=float)
    valid = arg > 0
    if np.any(valid):
        argv = arg[valid]
        out[valid] = A * (1 / np.sqrt(argv)) * np.exp(-(1 / (s * t ** (1/3))) * (1 / (argv ** 2))) * np.exp(-(argv ** 2) / s)
    out[~np.isfinite(out)] = np.nan
    return out

def nice_time_label(t):
    return f"t={t:,.0f}" if t >= 1000 else f"t={t:g}"

def compute_plot_payload(density, t_eval, rho, params, plot_every, scatter_every, offset, dxl):
    L = int(params["L"])
    dxl = int(max(10, min(dxl, L - 1)))
    plot_every = max(1, int(plot_every))
    scatter_every = max(1, int(scatter_every))
    offset = max(0, int(offset))
    plot_times = list(range(0, len(t_eval), plot_every)) or [0]
    xl = np.arange(-dxl, 0)
    z_collection, y_collection, times = [], [], []
    zmin, zmax = np.inf, -np.inf
    ymin, ymax = np.inf, -np.inf
    for ti in plot_times:
        t = t_eval[ti]
        y = density[:, ti]
        profile = rho - y[-dxl:]
        maxima_idx, _ = find_peaks(profile)
        if len(maxima_idx) == 0:
            continue
        start_idx = min(max(maxima_idx[0] - offset, 0), len(xl) - 1)
        z = xl[start_idx:] / np.power(t, 2 / 3)
        y_scaled = profile[start_idx:] * np.power(t, 1 / 3)
        z = z[::scatter_every]
        y_scaled = y_scaled[::scatter_every]
        valid = np.isfinite(z) & np.isfinite(y_scaled)
        z = z[valid]
        y_scaled = y_scaled[valid]
        if len(z) == 0:
            continue
        z_collection.append(np.asarray(z, dtype=float).tolist())
        y_collection.append(np.asarray(y_scaled, dtype=float).tolist())
        times.append(float(t))
        zmin, zmax = min(zmin, float(np.min(z))), max(zmax, float(np.max(z)))
        ymin, ymax = min(ymin, float(np.min(y_scaled))), max(ymax, float(np.max(y_scaled)))
    if not times:
        return {"empty": True, "message": "No peaks found for current settings."}
    default_v0, default_A, default_s = 1.237, 0.45, 0.68
    for z, t in zip(z_collection, times):
        m = model_curve(np.asarray(z, dtype=float), t, default_v0, default_A, default_s)
        valid = np.isfinite(m)
        if np.any(valid):
            ymin, ymax = min(ymin, float(np.min(m[valid]))), max(ymax, float(np.max(m[valid])))
    if abs(ymax-ymin) < 1e-12:
        ymin, ymax = ymin-1, ymax+1
    pad = 0.08 * (ymax - ymin)
    return {
        "empty": False,
        "times": times,
        "labels": [nice_time_label(t) for t in times],
        "z_collection": z_collection,
        "y_collection": y_collection,
        "x_range": [float(zmax), float(zmin)],
        "y_range": [float(ymin - pad), float(ymax + pad)],
        "title": f"L={params['L']:,}, rho={rho:.4f}, p={params['p']}, q={params['q']}, p'={params['p_s']}, q'={params['q_s']}, solver={params['solver']}"
    }


In [ ]:

style = {"description_width": "140px"}
layout_wide = widgets.Layout(width="340px")
layout_med = widgets.Layout(width="220px")
p_widget = widgets.FloatText(value=1.0, description="p", style=style, layout=layout_med)
q_widget = widgets.FloatText(value=0.1, description="q", style=style, layout=layout_med)
ps_widget = widgets.FloatText(value=0.1, description="p_s", style=style, layout=layout_med)
qs_widget = widgets.FloatText(value=1.0, description="q_s", style=style, layout=layout_med)
L_widget = widgets.IntText(value=1500, description="L", style=style, layout=layout_med)
rho_mode_widget = widgets.Dropdown(options=[("rho = rc1", "rc1"), ("rho = rc2", "rc2"), ("manual rho", "manual")], value="rc2", description="initial density", style=style, layout=layout_wide)
rho_manual_widget = widgets.FloatText(value=0.1, description="manual rho", style=style, layout=layout_med)
t_start_widget = widgets.FloatText(value=4e4, description="t start", style=style, layout=layout_med)
t_end_widget = widgets.FloatText(value=1e5, description="t end", style=style, layout=layout_med)
t_step_widget = widgets.FloatText(value=1e4, description="t step", style=style, layout=layout_med)
solver_widget = widgets.Dropdown(options=["LSODA", "BDF", "RK45"], value="LSODA", description="solver", style=style, layout=layout_med)
plot_every_widget = widgets.IntSlider(value=2, min=1, max=10, step=1, description="plot every", style=style, layout=layout_wide, continuous_update=False)
scatter_every_widget = widgets.IntSlider(value=35, min=1, max=200, step=1, description="scatter every", style=style, layout=layout_wide, continuous_update=False)
offset_widget = widgets.IntSlider(value=300, min=0, max=3000, step=10, description="offset", style=style, layout=layout_wide, continuous_update=False)
dxl_widget = widgets.IntText(value=1200, description="dxl", style=style, layout=layout_med)
run_button = widgets.Button(description="Run simulation", button_style="primary", icon="play")
status_html = widgets.HTML("<b>Status:</b> Ready.")
rho_info = widgets.HTML()
app_out = widgets.Output()

def current_rho_value():
    rc1, rc2 = get_rho_crit(p_widget.value, q_widget.value, ps_widget.value, qs_widget.value)
    if rho_mode_widget.value == "rc1": return rc1
    if rho_mode_widget.value == "rc2": return rc2
    return rho_manual_widget.value

def update_rho_help(*_):
    rc1, rc2 = get_rho_crit(p_widget.value, q_widget.value, ps_widget.value, qs_widget.value)
    selected = current_rho_value()
    rho_manual_widget.disabled = (rho_mode_widget.value != "manual")
    rho_info.value = f"<b>rho candidates:</b> rc1={rc1:.6g}, rc2={rc2:.6g} &nbsp; | &nbsp; <b>selected rho:</b> {selected:.6g}"

for w in [p_widget, q_widget, ps_widget, qs_widget, rho_mode_widget, rho_manual_widget]:
    w.observe(update_rho_help, names="value")
update_rho_help()


In [ ]:

def build_client_side_html(payload):
    payload_json = json.dumps(payload)
    div_id = f"fitapp_{uuid.uuid4().hex}"
    template = r"""
<div id="__DIVID__" style="font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; max-width: 1200px;">
  <style>
    #__DIVID__ .row { display:flex; flex-wrap:wrap; gap:14px; align-items:center; margin:8px 0; }
    #__DIVID__ .panel { background:#fafafa; border:1px solid #ddd; border-radius:10px; padding:12px 14px; margin:12px 0; }
    #__DIVID__ .control { min-width:260px; flex:1 1 260px; }
    #__DIVID__ label { display:block; font-weight:600; margin-bottom:4px; }
    #__DIVID__ input[type='range'] { width:100%; }
    #__DIVID__ .value { display:inline-block; min-width:72px; font-variant-numeric:tabular-nums; margin-left:10px; }
    #__DIVID__ .title { font-size:22px; font-weight:700; margin-bottom:6px; }
    #__DIVID__ .subtitle { color:#444; margin-bottom:8px; }
    #__DIVID__ .legend { display:flex; flex-wrap:wrap; gap:12px; margin-top:10px; font-size:13px; }
    #__DIVID__ .legend-item { display:inline-flex; align-items:center; gap:6px; }
    #__DIVID__ .swatch { width:12px; height:12px; border-radius:999px; display:inline-block; }
    #__DIVID__ canvas { width:100%; max-width:1100px; height:auto; border:1px solid #ccc; border-radius:10px; background:white; display:block; }
    #__DIVID__ .hint { color:#555; font-size:14px; }
  </style>
  <div class="title">Single-file dynamics — smooth fit explorer</div>
  <div class="subtitle">__TITLE__</div>
  <div class="panel">
    <div class="row">
      <div class="control"><label for="__DIVID___v0">v0 <span class="value" id="__DIVID___v0_val">1.237</span></label><input id="__DIVID___v0" type="range" min="0.001" max="10" step="0.001" value="1.237"></div>
      <div class="control"><label for="__DIVID___A">A <span class="value" id="__DIVID___A_val">0.450</span></label><input id="__DIVID___A" type="range" min="0.001" max="10" step="0.001" value="0.45"></div>
      <div class="control"><label for="__DIVID___s">s <span class="value" id="__DIVID___s_val">0.680</span></label><input id="__DIVID___s" type="range" min="0.001" max="10" step="0.001" value="0.68"></div>
    </div>
    <div class="hint">These sliders update <b>entirely in the browser</b>. No Python callback is triggered while you drag them.</div>
  </div>
  <canvas id="__DIVID___canvas" width="1100" height="680"></canvas>
  <div class="legend" id="__DIVID___legend"></div>
</div>
<script>
(() => {
  const payload = __PAYLOAD__;
  const canvas = document.getElementById("__DIVID___canvas");
  const ctx = canvas.getContext("2d");
  const v0Input = document.getElementById("__DIVID___v0");
  const AInput = document.getElementById("__DIVID___A");
  const sInput = document.getElementById("__DIVID___s");
  const v0Val = document.getElementById("__DIVID___v0_val");
  const AVal = document.getElementById("__DIVID___A_val");
  const sVal = document.getElementById("__DIVID___s_val");
  const legend = document.getElementById("__DIVID___legend");
  const traces = payload.z_collection.map((z, i) => ({ z, y: payload.y_collection[i], t: payload.times[i], label: payload.labels[i] }));
  const colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#17becf", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22"];
  legend.innerHTML = traces.map((tr, i) => `<span class="legend-item"><span class="swatch" style="background:${colors[i % colors.length]}"></span><span>${tr.label}</span></span>`).join("");
  const W = canvas.width, H = canvas.height;
  const margin = { left: 88, right: 24, top: 30, bottom: 72 };
  const plotW = W - margin.left - margin.right, plotH = H - margin.top - margin.bottom;
  const xMin = payload.x_range[1], xMax = payload.x_range[0], yMin = payload.y_range[0], yMax = payload.y_range[1];
  function sx(x){ const frac=(x-xMin)/(xMax-xMin); return margin.left + frac*plotW; }
  function sy(y){ const frac=(y-yMin)/(yMax-yMin); return H-margin.bottom-frac*plotH; }
  function modelY(z,t,v0,A,s){ const arg=v0+(2*z/3); if(!(arg>0)) return NaN; const out=(A/Math.sqrt(arg))*Math.exp(-(1/(s*Math.pow(t,1/3)))*(1/(arg*arg)))*Math.exp(-(arg*arg)/s); return Number.isFinite(out)?out:NaN; }
  function niceNum(x){ return Number(x).toFixed(3); }
  let rafPending=false;
  function drawAxes(){
    ctx.save(); ctx.clearRect(0,0,W,H); ctx.fillStyle='white'; ctx.fillRect(0,0,W,H); ctx.strokeStyle='#222'; ctx.lineWidth=1.2; ctx.strokeRect(margin.left, margin.top, plotW, plotH);
    ctx.strokeStyle='#e6e6e6'; ctx.lineWidth=1; const nTicks=6;
    for(let i=0;i<=nTicks;i++){ const x=margin.left+(i/nTicks)*plotW; ctx.beginPath(); ctx.moveTo(x,margin.top); ctx.lineTo(x,H-margin.bottom); ctx.stroke(); const y=margin.top+(i/nTicks)*plotH; ctx.beginPath(); ctx.moveTo(margin.left,y); ctx.lineTo(W-margin.right,y); ctx.stroke(); }
    ctx.fillStyle='#222'; ctx.font='13px system-ui, sans-serif'; ctx.textAlign='center'; ctx.textBaseline='top';
    for(let i=0;i<=nTicks;i++){ const frac=i/nTicks; const x=margin.left+frac*plotW; const val=xMin+frac*(xMax-xMin); ctx.fillText(val.toFixed(2), x, H-margin.bottom+10); }
    ctx.textAlign='right'; ctx.textBaseline='middle';
    for(let i=0;i<=nTicks;i++){ const frac=i/nTicks; const y=H-margin.bottom-frac*plotH; const val=yMin+frac*(yMax-yMin); ctx.fillText(val.toFixed(2), margin.left-10, y); }
    ctx.font='16px system-ui, sans-serif'; ctx.textAlign='center'; ctx.textBaseline='alphabetic'; ctx.fillText('z', margin.left+plotW/2, H-24);
    ctx.save(); ctx.translate(24, margin.top+plotH/2); ctx.rotate(-Math.PI/2); ctx.fillText('[ρ̄ − ρ(z)] t^(1/3)', 0, 0); ctx.restore(); ctx.restore();
  }
  function drawData(){ traces.forEach((tr,i)=>{ ctx.fillStyle=colors[i%colors.length]; for(let k=0;k<tr.z.length;k++){ const x=sx(tr.z[k]), y=sy(tr.y[k]); ctx.beginPath(); ctx.arc(x,y,2.4,0,2*Math.PI); ctx.fill(); } }); }
  function drawModel(v0,A,s){ traces.forEach((tr,i)=>{ ctx.strokeStyle=colors[i%colors.length]; ctx.lineWidth=2.2; ctx.beginPath(); let started=false; for(let k=0;k<tr.z.length;k++){ const y=modelY(tr.z[k],tr.t,v0,A,s); if(!Number.isFinite(y)){ started=false; continue; } const xpx=sx(tr.z[k]), ypx=sy(y); if(!started){ ctx.moveTo(xpx, ypx); started=true; } else { ctx.lineTo(xpx, ypx); } } ctx.stroke(); }); }
  function drawNow(){ rafPending=false; const v0=parseFloat(v0Input.value), A=parseFloat(AInput.value), s=parseFloat(sInput.value); v0Val.textContent=niceNum(v0); AVal.textContent=niceNum(A); sVal.textContent=niceNum(s); drawAxes(); drawData(); drawModel(v0,A,s); }
  function requestDraw(){ if(!rafPending){ rafPending=true; requestAnimationFrame(drawNow); } }
  [v0Input, AInput, sInput].forEach(el=>el.addEventListener('input', requestDraw));
  drawNow();
})();
</script>
"""
    return template.replace('__DIVID__', div_id).replace('__TITLE__', html.escape(payload['title'])).replace('__PAYLOAD__', payload_json)


def run_simulation(_=None):
    status_html.value = "<b>Status:</b> Running simulation..."
    with app_out:
        clear_output()
        display(HTML("<div style='padding:10px;font-family:system-ui'>Running simulation...</div>"))
    try:
        p,q,p_s,q_s = float(p_widget.value), float(q_widget.value), float(ps_widget.value), float(qs_widget.value)
        L = int(L_widget.value)
        rho = float(current_rho_value())
        solver = solver_widget.value
        t_start,t_end,t_step = float(t_start_widget.value), float(t_end_widget.value), float(t_step_widget.value)
        if L < 10: raise ValueError('L must be at least 10.')
        if not np.isfinite(rho): raise ValueError('Selected rho is not finite.')
        if t_step <= 0 or t_end <= t_start: raise ValueError('Need t_step > 0 and t_end > t_start.')
        t_eval = np.arange(t_start, t_end + 0.5*t_step, t_step)
        phi0 = np.full(L, rho, dtype=float)
        sol = solve_ivp(dphi_dt, [float(np.min(t_eval)), float(np.max(t_eval))], phi0, method=solver, t_eval=t_eval, args=(p,q,p_s,q_s))
        payload = compute_plot_payload(sol.y, t_eval, rho, {"p":p, "q":q, "p_s":p_s, "q_s":q_s, "L":L, "solver":solver}, plot_every_widget.value, scatter_every_widget.value, offset_widget.value, dxl_widget.value)
        if payload.get('empty'):
            status_html.value = "<b>Status:</b> Simulation finished, but no peaks were found."
            with app_out:
                clear_output(); display(HTML(f"<div style='padding:10px;font-family:system-ui'>{html.escape(payload['message'])}</div>"))
            return
        status_html.value = "<b>Status:</b> Ready. Drag the fit sliders below the plot."
        with app_out:
            clear_output(); display(HTML(build_client_side_html(payload)))
    except Exception as e:
        status_html.value = f"<b>Status:</b> Error: {html.escape(str(e))}"
        with app_out:
            clear_output(); display(HTML(f"<div style='padding:10px;color:#b00020;font-family:system-ui'><b>Error:</b> {html.escape(str(e))}</div>"))


In [ ]:

run_button.on_click(run_simulation)
controls = widgets.VBox([
    widgets.HTML("<h2>Single-file dynamics Voilà app</h2>"),
    widgets.HTML("Simulation parameters are handled in Python. Once the data is prepared, the fit sliders under the plot update <b>client-side</b> for much smoother interaction."),
    widgets.HBox([p_widget, q_widget, ps_widget, qs_widget, L_widget]),
    widgets.HBox([rho_mode_widget, rho_manual_widget]),
    rho_info,
    widgets.HBox([t_start_widget, t_end_widget, t_step_widget, solver_widget]),
    widgets.HBox([plot_every_widget, scatter_every_widget]),
    widgets.HBox([offset_widget, dxl_widget]),
    run_button,
    status_html,
    app_out,
])
display(controls)
